In [0]:
# ============================================================
# mt_context_builder — Agent Context Builder
# ============================================================

# Derive from mt_config (loaded before this notebook)
SCHEMA = f"{MT_CATALOG}.{MT_SCHEMA}"



def _load_technical_schema():
    rows = spark.read.table(f"{SCHEMA}.mt_information_schema").orderBy("table_name", "column_name").collect()
    lines = []
    current_table = None
    for r in rows:
        if r.table_name != current_table:
            current_table = r.table_name
            lines.append(f"\nTABLE: {SCHEMA}.{current_table}")
        jk = f" [JOIN KEY: {r.possible_join_group}]" if r.is_join_key else ""
        ps = " [PSEUDONYMIZED]" if r.is_pseudonymized else ""
        desc = f" -- {r.description}" if getattr(r, 'description', None) else ""
        lines.append(f"  {r.column_name} ({r.data_type}){jk}{ps}{desc}")
    return "\n".join(lines)


def _load_technical_join_map():
    """Build a clean join map showing only cross-table canonical joins (no within-table aliases)."""
    rows = spark.read.table(f"{SCHEMA}.mt_information_schema").filter("is_join_key = true").collect()
    groups = {}
    for r in rows:
        grp = r.possible_join_group or "unknown"
        if grp not in groups:
            groups[grp] = []
        groups[grp].append((r.table_name, r.column_name, getattr(r, 'description', '') or ''))
    # Only show one canonical column per table per group (skip ALIAS columns)
    lines = ["JOIN MAP (use these columns to join tables):"]
    for grp, entries in sorted(groups.items()):
        canonical = []
        seen_tables = set()
        for tbl, col, desc in entries:
            if tbl in seen_tables:
                continue  # skip aliases within same table
            if 'ALIAS' in desc.upper():
                continue  # skip explicitly marked aliases
            canonical.append(f"{tbl}.{col}")
            seen_tables.add(tbl)
        if canonical:
            lines.append(f"  {grp}: {' = '.join(canonical)}")
    # Add explicit join examples for common patterns
    lines.append("\nCOMMON JOIN PATTERNS:")
    lines.append("  User→Company: mt_safe_user.assignment_id1 → mt_safe_assignment.assignment_ID → mt_safe_assignment.company_ID1 → mt_safe_company.company_ID")
    lines.append("  User→CompanyType: ...→ mt_safe_company.subTypeId → mt_safe_company_type.type_id")
    lines.append("  User→Course: mt_safe_user.course_id → mt_safe_course.course_id1")
    lines.append("  On-time check: mt_safe_user.completedOn_ts <= mt_safe_assignment.deadline_assignment")
    return "\n".join(lines)


def _load_semantic_table_metadata():
    rows = spark.read.table(f"{SCHEMA}.mt_semantic_table_metadata").collect()
    lines = ["TABLE DESCRIPTIONS:"]
    for r in rows:
        lines.append(f"\n  {r.table_name} — {r.business_name}")
        lines.append(f"    {r.business_description}")
        lines.append(f"    Grain: {r.grain} | Key: {r.business_key}")
    return "\n".join(lines)


def _load_semantic_column_metadata():
    rows = spark.read.table(f"{SCHEMA}.mt_semantic_column_metadata").orderBy("table_name", "column_name").collect()
    lines = ["COLUMN DESCRIPTIONS:"]
    current_table = None
    for r in rows:
        if r.table_name != current_table:
            current_table = r.table_name
            lines.append(f"\n  {current_table}:")
        syn = f" (synonyms: {r.synonyms})" if r.synonyms else ""
        lines.append(f"    {r.column_name} = {r.business_name}: {r.description}{syn}")
    return "\n".join(lines)


def _load_metric_definitions():
    rows = spark.read.table(f"{SCHEMA}.mt_semantic_metric_definitions").filter("mapping_status = 'mapped'").collect()
    lines = ["METRIC DEFINITIONS:"]
    for r in rows:
        lines.append(f"\n  {r.metric_id}: {r.metric_name}")
        lines.append(f"    Definition: {r.business_definition}")
        lines.append(f"    Formula: {r.aggregation_logic}")
        lines.append(f"    Source: {r.source_table}")
        if r.notes:
            lines.append(f"    Note: {r.notes}")
    return "\n".join(lines)


def _load_semantic_join_rules():
    rows = spark.read.table(f"{SCHEMA}.mt_semantic_join_rules").collect()
    lines = ["JOIN RULES:"]
    for r in rows:
        lines.append(f"  {r.left_table}.{r.left_column} {r.join_type} JOIN {r.right_table}.{r.right_column} ({r.cardinality}) — {r.description}")
    return "\n".join(lines)


def _load_business_rules():
    rows = spark.read.table(f"{SCHEMA}.mt_semantic_business_rules").filter("mapping_status = 'mapped'").collect()
    lines = ["BUSINESS RULES:"]
    for r in rows:
        lines.append(f"\n  {r.rule_id}: {r.rule_name} ({r.rule_category})")
        lines.append(f"    {r.description}")
        lines.append(f"    Logic: {r.rule_logic}")
    return "\n".join(lines)


print("✓ mt_context_builder helpers loaded")
print(f"  Tables: {len(APPROVED_SAFE_TABLES)}")

In [0]:
# ============================================================
# UNIFIED CONTEXT BUILDER — Architecture-Aware
# ============================================================
# Usage:
#   ctx = build_unified_context("SAS")         # full semantic in context
#   ctx = build_unified_context("SAS_RAG")     # base context only (semantic retrieved)
#   prompt = format_unified_context_for_prompt(ctx)
# ============================================================

from typing import Literal

# Format SQL_SAFETY_RULES (list from mt_config) as prompt-ready string
_SQL_RULES_TEXT = "SQL SAFETY RULES:\n" + "\n".join(f"{i+1}. {r}" for i, r in enumerate(SQL_SAFETY_RULES))

# Architecture-aware semantic delivery rules
_SEMANTIC_DIRECT_ARCHITECTURES = ["SAS"]
_SEMANTIC_RETRIEVED_ARCHITECTURES = ["SAS_RAG", "MAS_RAG"]


def build_unified_context(architecture: str) -> dict:
    """
    Build the agent context for a given architecture under UNIFIED context mode.

    For direct-semantic architectures (SAS):
        Returns technical schema + full semantic layer in the context dict.
    For RAG architectures (SAS_RAG, MAS_RAG):
        Returns technical schema only. Semantic content retrieved separately.

    Args:
        architecture: one of SAS, SAS_RAG, MAS_RAG

    Returns:
        dict with context_mode, architecture, semantic_delivery, and content fields.
    """
    all_known = _SEMANTIC_DIRECT_ARCHITECTURES + _SEMANTIC_RETRIEVED_ARCHITECTURES
    if architecture not in all_known:
        raise ValueError(f"Unknown architecture: {architecture}. Valid: {all_known}")

    is_direct = architecture in _SEMANTIC_DIRECT_ARCHITECTURES

    ctx = {
        "context_mode":        "UNIFIED",
        "architecture":        architecture,
        "semantic_delivery":   "direct" if is_direct else "retrieved",
        "approved_tables":     APPROVED_SAFE_TABLES,
        "technical_schema":    _load_technical_schema(),
        "technical_join_map":  _load_technical_join_map(),
        "sql_safety_rules":    _SQL_RULES_TEXT,
    }

    if is_direct:
        # SAS architectures: inject full semantic layer directly
        ctx["semantic_table_metadata"]  = _load_semantic_table_metadata()
        ctx["semantic_column_metadata"] = _load_semantic_column_metadata()
        ctx["metric_definitions"]       = _load_metric_definitions()
        ctx["semantic_join_rules"]      = _load_semantic_join_rules()
        ctx["business_rules"]           = _load_business_rules()
    else:
        # RAG architectures: semantic retrieved separately
        ctx["semantic_table_metadata"]  = None
        ctx["semantic_column_metadata"] = None
        ctx["metric_definitions"]       = None
        ctx["semantic_join_rules"]      = None
        ctx["business_rules"]           = None

    return ctx


def format_unified_context_for_prompt(ctx: dict, include_semantic: bool = True) -> str:
    """
    Format unified context dict into a single LLM-ready string.

    Args:
        ctx: context dict from build_unified_context()
        include_semantic: if False, omit semantic sections even if present

    Returns:
        formatted prompt string
    """
    sections = []

    # Header
    sections.append(f"CONTEXT MODE: {ctx.get('context_mode', 'UNIFIED')}")
    sections.append(f"ARCHITECTURE: {ctx.get('architecture', 'UNKNOWN')}")
    sections.append(f"SEMANTIC DELIVERY: {ctx.get('semantic_delivery', 'unknown')}")

    # Approved tables
    sections.append("\nAPPROVED TABLES (agents may only query these):")
    sections.append("\n".join(f"  - {t}" for t in ctx.get('approved_tables', [])))

    # SQL safety rules
    sections.append(f"\n{ctx.get('sql_safety_rules', '')}")

    # Technical schema
    sections.append("\nTECHNICAL SCHEMA:")
    sections.append(ctx.get('technical_schema', ''))

    # Join map
    sections.append(f"\n{ctx.get('technical_join_map', '')}")

    # Semantic layer (only for direct-delivery architectures or if include_semantic=True)
    if include_semantic:
        if ctx.get('semantic_table_metadata'):
            sections.append(f"\n{ctx['semantic_table_metadata']}")
        if ctx.get('semantic_column_metadata'):
            sections.append(f"\n{ctx['semantic_column_metadata']}")
        if ctx.get('metric_definitions'):
            sections.append(f"\n{ctx['metric_definitions']}")
        if ctx.get('semantic_join_rules'):
            sections.append(f"\n{ctx['semantic_join_rules']}")
        if ctx.get('business_rules'):
            sections.append(f"\n{ctx['business_rules']}")

    return "\n".join(sections)


def get_semantic_retrieval_source() -> str:
    """
    Returns a combined text of all semantic layer content for RAG indexing/retrieval.
    Use this to build embeddings for RAG architectures.
    """
    parts = [
        _load_semantic_table_metadata(),
        _load_semantic_column_metadata(),
        _load_metric_definitions(),
        _load_semantic_join_rules(),
        _load_business_rules(),
    ]
    return "\n\n".join([p for p in parts if p])


print("\u2713 mt_context_builder loaded (UNIFIED mode)")
print("  build_unified_context(architecture)        — SAS=direct, RAG=retrieved")
print("  format_unified_context_for_prompt(ctx)    — LLM-ready prompt string")
print("  get_semantic_retrieval_source()           — semantic content for RAG indexing")